# SML Assignment 4
## Q1: AdaBoost | Q2: Gradient Boosting | Q3: Perceptron

> **Note:** Place `mnist.npz` one directory above this notebook (i.e., `../mnist.npz`) before running Q1 and Q2.

---
# Q1 — AdaBoost with Decision Stumps on MNIST (Classes 4 vs 9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Set a random seed for reproducibility
np.random.seed(42)

In [ ]:
def load_and_preprocess_data(file_path):
    print("Loading data...")
    with np.load(file_path) as data:
        x_train, y_train = data['x_train'], data['y_train']
        x_test, y_test = data['x_test'], data['y_test']
    
    # Flatten images
    x_train = x_train.reshape(x_train.shape[0], -1)
    x_test = x_test.reshape(x_test.shape[0], -1)
    
    # Normalize pixel values
    x_train = x_train.astype(np.float64) / 255.0
    x_test = x_test.astype(np.float64) / 255.0
    
    # Filter classes 4 and 9
    train_mask = (y_train == 4) | (y_train == 9)
    x_train_filtered = x_train[train_mask]
    y_train_filtered = y_train[train_mask]
    
    test_mask = (y_test == 4) | (y_test == 9)
    x_test_filtered = x_test[test_mask]
    y_test_filtered = y_test[test_mask]
    
    # Relabel 4 -> -1, 9 -> 1
    y_train_binary = np.where(y_train_filtered == 9, 1, -1)
    y_test_binary = np.where(y_test_filtered == 9, 1, -1)
    
    # Validation split: keep aside 1000 from each class
    idx_4 = np.where(y_train_binary == -1)[0]
    idx_9 = np.where(y_train_binary == 1)[0]
    
    np.random.shuffle(idx_4)
    np.random.shuffle(idx_9)
    
    val_idx_4 = idx_4[:1000]
    val_idx_9 = idx_9[:1000]
    train_idx_4 = idx_4[1000:]
    train_idx_9 = idx_9[1000:]
    
    val_indices = np.concatenate([val_idx_4, val_idx_9])
    train_indices = np.concatenate([train_idx_4, train_idx_9])
    
    np.random.shuffle(val_indices)
    np.random.shuffle(train_indices)
    
    x_val = x_train_filtered[val_indices]
    y_val = y_train_binary[val_indices]
    x_train_final = x_train_filtered[train_indices]
    y_train_final = y_train_binary[train_indices]
    
    print(f"Train set: {x_train_final.shape}")
    print(f"Validation set: {x_val.shape}")
    print(f"Test set: {x_test_filtered.shape}")
    
    return x_train_final, y_train_final, x_val, y_val, x_test_filtered, y_test_binary

In [ ]:
def apply_pca(x_train, x_val, x_test, n_components=5):
    print(f"Applying PCA (reducing to {n_components} dimensions)...")
    mean = np.mean(x_train, axis=0)
    x_train_centered = x_train - mean
    cov_matrix = np.cov(x_train_centered, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    sorted_idx = np.argsort(eigenvalues)[::-1]
    eigenvectors = eigenvectors[:, sorted_idx]
    pca_matrix = eigenvectors[:, :n_components]
    x_train_pca = np.dot(x_train - mean, pca_matrix)
    x_val_pca = np.dot(x_val - mean, pca_matrix)
    x_test_pca = np.dot(x_test - mean, pca_matrix)
    return x_train_pca, x_val_pca, x_test_pca

In [ ]:
class DecisionStump:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.polarity = 1
        self.alpha = None
        
    def fit(self, X, y, weights):
        n_samples, n_features = X.shape
        min_error = float('inf')
        
        for feature_idx in range(n_features):
            feature_values = X[:, feature_idx]
            unique_vals = np.unique(feature_values)
            sorted_unique = np.sort(unique_vals)
            
            if len(sorted_unique) > 1:
                midpoints = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0
                if len(midpoints) > 1000:
                    midpoints = np.random.choice(midpoints, 1000, replace=False)
            else:
                midpoints = sorted_unique
            
            if len(midpoints) == 0:
                continue
            
            less_than = feature_values[:, np.newaxis] < midpoints[np.newaxis, :]
            pred_1 = np.where(less_than, -1, 1)
            mismatch_1 = pred_1 != y[:, np.newaxis]
            error_1 = np.sum(weights[:, np.newaxis] * mismatch_1, axis=0)
            error_neg_1 = np.sum(weights) - error_1
            
            best_idx_1 = np.argmin(error_1)
            min_err_1 = error_1[best_idx_1]
            best_idx_neg_1 = np.argmin(error_neg_1)
            min_err_neg_1 = error_neg_1[best_idx_neg_1]
            
            if min_err_1 < min_error:
                min_error = min_err_1
                self.feature_index = feature_idx
                self.threshold = midpoints[best_idx_1]
                self.polarity = 1
            if min_err_neg_1 < min_error:
                min_error = min_err_neg_1
                self.feature_index = feature_idx
                self.threshold = midpoints[best_idx_neg_1]
                self.polarity = -1
                
        return min_error

    def predict(self, X):
        n_samples = X.shape[0]
        predictions = np.ones(n_samples)
        feature_values = X[:, self.feature_index]
        if self.polarity == 1:
            predictions[feature_values < self.threshold] = -1
        else:
            predictions[feature_values >= self.threshold] = -1
        return predictions

In [ ]:
def adaboost(X_train, y_train, X_val, y_val, n_estimators=300):
    print(f"Training AdaBoost with {n_estimators} stumps...")
    n_samples = X_train.shape[0]
    weights = np.ones(n_samples) / n_samples
    stumps = []
    val_accuracies = []
    val_preds = np.zeros(X_val.shape[0])
    
    for i in range(n_estimators):
        stump = DecisionStump()
        error = stump.fit(X_train, y_train, weights)
        eps = 1e-10
        alpha = 0.5 * np.log((1.0 - error + eps) / (error + eps))
        stump.alpha = alpha
        predictions = stump.predict(X_train)
        weights = weights * np.exp(-alpha * y_train * predictions)
        weights /= np.sum(weights)
        stumps.append(stump)
        val_preds += alpha * stump.predict(X_val)
        val_acc = np.mean(np.where(val_preds >= 0, 1, -1) == y_val)
        val_accuracies.append(val_acc)
        if (i+1) % 30 == 0:
            print(f"Tree {i+1}/{n_estimators}, Validation Accuracy: {val_acc:.4f}")
            
    return stumps, val_accuracies

In [ ]:
# --- Q1 Main Execution ---
file_path = '../mnist.npz'

if not os.path.exists(file_path):
    print(f"Error: {file_path} not found. Please place mnist.npz one directory above.")
else:
    x_train, y_train, x_val, y_val, x_test, y_test = load_and_preprocess_data(file_path)
    x_train_pca, x_val_pca, x_test_pca = apply_pca(x_train, x_val, x_test, n_components=5)
    stumps, val_accuracies = adaboost(x_train_pca, y_train, x_val_pca, y_val, n_estimators=300)

    best_iter = np.argmax(val_accuracies)
    print(f"\nBest iteration: {best_iter + 1} with validation accuracy: {val_accuracies[best_iter]:.4f}")

    # Test set evaluation
    test_preds = np.zeros(x_test_pca.shape[0])
    for i in range(best_iter + 1):
        test_preds += stumps[i].alpha * stumps[i].predict(x_test_pca)
    final_test_preds = np.where(test_preds >= 0, 1, -1)
    test_accuracy = np.mean(final_test_preds == y_test)
    print(f"Test Accuracy: {test_accuracy:.4f}")

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, 301), val_accuracies, label='Validation Accuracy', color='b')
    plt.axvline(x=best_iter + 1, color='r', linestyle='--', label=f'Best Iteration ({best_iter + 1})')
    plt.xlabel('Number of Trees')
    plt.ylabel('Validation Accuracy')
    plt.title('AdaBoost Validation Accuracy vs Number of Trees')
    plt.legend()
    plt.grid(True)
    plt.savefig('q1_val_accuracy.png')
    plt.show()
    print("Plot saved to q1_val_accuracy.png")

---
# Q2 — Gradient Boosting with Absolute Loss on MNIST (Classes 4 vs 9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

np.random.seed(42)

In [ ]:
def load_and_preprocess_data_q2(file_path):
    print("Loading data...")
    with np.load(file_path) as data:
        x_train, y_train = data['x_train'], data['y_train']
        x_test, y_test = data['x_test'], data['y_test']
    
    x_train = x_train.reshape(x_train.shape[0], -1)
    x_test = x_test.reshape(x_test.shape[0], -1)
    x_train = x_train.astype(np.float64) / 255.0
    x_test = x_test.astype(np.float64) / 255.0
    
    train_mask = (y_train == 4) | (y_train == 9)
    x_train_filtered = x_train[train_mask]
    y_train_filtered = y_train[train_mask]
    test_mask = (y_test == 4) | (y_test == 9)
    x_test_filtered = x_test[test_mask]
    y_test_filtered = y_test[test_mask]
    
    y_train_binary = np.where(y_train_filtered == 9, 1, -1)
    y_test_binary = np.where(y_test_filtered == 9, 1, -1)
    
    idx_4 = np.where(y_train_binary == -1)[0]
    idx_9 = np.where(y_train_binary == 1)[0]
    np.random.shuffle(idx_4)
    np.random.shuffle(idx_9)
    
    val_indices = np.concatenate([idx_4[:1000], idx_9[:1000]])
    train_indices = np.concatenate([idx_4[1000:], idx_9[1000:]])
    np.random.shuffle(val_indices)
    np.random.shuffle(train_indices)
    
    x_val = x_train_filtered[val_indices]
    y_val = y_train_binary[val_indices]
    x_train_final = x_train_filtered[train_indices]
    y_train_final = y_train_binary[train_indices]
    
    return x_train_final, y_train_final, x_val, y_val, x_test_filtered, y_test_binary


def apply_pca_q2(x_train, x_val, x_test, n_components=5):
    print(f"Applying PCA (reducing to {n_components} dimensions)...")
    mean = np.mean(x_train, axis=0)
    cov_matrix = np.cov(x_train - mean, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    sorted_idx = np.argsort(eigenvalues)[::-1]
    eigenvectors = eigenvectors[:, sorted_idx]
    pca_matrix = eigenvectors[:, :n_components]
    x_train_pca = np.dot(x_train - mean, pca_matrix)
    x_val_pca = np.dot(x_val - mean, pca_matrix)
    x_test_pca = np.dot(x_test - mean, pca_matrix)
    return x_train_pca, x_val_pca, x_test_pca

In [ ]:
class RegressionStump:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.c_left = None
        self.c_right = None
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        best_score = -float('inf')
        S_total = np.sum(y)
        
        for feature_idx in range(n_features):
            feature_values = X[:, feature_idx]
            unique_vals = np.unique(feature_values)
            sorted_unique = np.sort(unique_vals)
            
            if len(sorted_unique) > 1:
                midpoints = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0
                if len(midpoints) > 1000:
                    midpoints = np.random.choice(midpoints, 1000, replace=False)
            else:
                midpoints = sorted_unique
            
            if len(midpoints) == 0:
                continue
            
            less_than = feature_values[:, np.newaxis] < midpoints[np.newaxis, :]
            N_L = np.sum(less_than, axis=0)
            N_R = n_samples - N_L
            S_L = y @ less_than
            S_R = S_total - S_L
            score = (S_L**2) / np.maximum(N_L, 1) + (S_R**2) / np.maximum(N_R, 1)
            
            best_idx = np.argmax(score)
            if score[best_idx] > best_score:
                best_score = score[best_idx]
                self.feature_index = feature_idx
                self.threshold = midpoints[best_idx]
                n_l = N_L[best_idx]; n_r = N_R[best_idx]
                s_l = S_L[best_idx]; s_r = S_R[best_idx]
                self.c_left = s_l / n_l if n_l > 0 else 0
                self.c_right = s_r / n_r if n_r > 0 else 0
                
    def predict(self, X):
        feature_values = X[:, self.feature_index]
        return np.where(feature_values < self.threshold, self.c_left, self.c_right)

In [ ]:
def gradient_boosting(X_train, y_train, X_val, y_val, n_estimators=300, learning_rate=0.01):
    stumps = []
    val_mses = []
    train_preds = np.zeros(X_train.shape[0])
    val_preds = np.zeros(X_val.shape[0])
    labels = y_train.copy()
    
    for i in range(n_estimators):
        stump = RegressionStump()
        stump.fit(X_train, labels)
        train_preds += learning_rate * stump.predict(X_train)
        val_preds += learning_rate * stump.predict(X_val)
        labels = np.sign(y_train - train_preds)  # Absolute loss pseudo-residuals
        stumps.append(stump)
        val_mse = np.mean((y_val - val_preds)**2)
        val_mses.append(val_mse)
        
    return stumps, val_mses

In [ ]:
def run_experiment(x_train, y_train, x_val, y_val, x_test, y_test,
                   learning_rates=[0.001, 0.01, 0.1, 0.2, 0.5, 1]):
    results = {}
    plt.figure(figsize=(10, 6))
    
    for lr in learning_rates:
        print(f"Training Gradient Boosting with learning rate {lr}...")
        stumps, val_mses = gradient_boosting(x_train, y_train, x_val, y_val,
                                             n_estimators=300, learning_rate=lr)
        best_iter = np.argmin(val_mses)
        print(f"  LR={lr}: Best iteration: {best_iter + 1}, Val MSE: {val_mses[best_iter]:.4f}")
        
        test_preds = np.zeros(x_test.shape[0])
        for i in range(best_iter + 1):
            test_preds += lr * stumps[i].predict(x_test)
        test_mse = np.mean((y_test - test_preds)**2)
        print(f"  LR={lr}: Test MSE: {test_mse:.4f}")
        
        results[lr] = {'val_mses': val_mses, 'test_mse': test_mse, 'best_iter': best_iter + 1}
        plt.plot(range(1, 301), val_mses, label=f'LR = {lr}')
        
    plt.xlabel('Number of Trees')
    plt.ylabel('Validation MSE')
    plt.title('Validation MSE vs Number of Trees for different Learning Rates')
    plt.legend()
    plt.grid(True)
    plt.savefig('q2_val_mse_all.png')
    plt.show()
    print("Plot saved to q2_val_mse_all.png")
    return results

In [ ]:
# --- Q2 Main Execution ---
file_path = '../mnist.npz'

if not os.path.exists(file_path):
    print(f"Error: {file_path} not found. Please place mnist.npz one directory above.")
else:
    x_train2, y_train2, x_val2, y_val2, x_test2, y_test2 = load_and_preprocess_data_q2(file_path)
    x_train_pca2, x_val_pca2, x_test_pca2 = apply_pca_q2(x_train2, x_val2, x_test2, n_components=5)

    # Part 1: eta = 0.01 only
    print("--- Running for eta = 0.01 ---")
    stumps_01, val_mses_01 = gradient_boosting(x_train_pca2, y_train2, x_val_pca2, y_val2,
                                                n_estimators=300, learning_rate=0.01)
    best_iter_01 = np.argmin(val_mses_01)

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, 301), val_mses_01, color='b')
    plt.axvline(x=best_iter_01 + 1, color='r', linestyle='--',
                label=f'Best Iteration ({best_iter_01 + 1})')
    plt.xlabel('Number of Trees')
    plt.ylabel('Validation MSE')
    plt.title('Validation MSE vs Number of Trees (eta = 0.01)')
    plt.legend()
    plt.grid(True)
    plt.savefig('q2_val_mse_0.01.png')
    plt.show()

    test_preds_01 = np.zeros(x_test_pca2.shape[0])
    for i in range(best_iter_01 + 1):
        test_preds_01 += 0.01 * stumps_01[i].predict(x_test_pca2)
    test_mse_01 = np.mean((y_test2 - test_preds_01)**2)
    print(f"Test MSE for eta=0.01: {test_mse_01:.4f}\n")

    # Part 2: All learning rates
    print("--- Running for all learning rates ---")
    results = run_experiment(x_train_pca2, y_train2, x_val_pca2, y_val2, x_test_pca2, y_test2)

---
# Q3 — Perceptron on Synthetic Gaussian Datasets (A and B)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
def generate_dataset(mean1, cov1, mean2, cov2, n_samples=200):
    X1 = np.random.multivariate_normal(mean1, cov1, n_samples)
    y1 = -np.ones(n_samples)
    X2 = np.random.multivariate_normal(mean2, cov2, n_samples)
    y2 = np.ones(n_samples)
    X = np.vstack((X1, X2))
    y = np.concatenate((y1, y2))
    indices = np.arange(2 * n_samples)
    np.random.shuffle(indices)
    return X[indices], y[indices]

In [ ]:
class Perceptron:
    def __init__(self, learning_rate=0.01, max_epochs=300):
        self.learning_rate = learning_rate
        self.max_epochs = max_epochs
        self.weights = None
        self.bias = None
        self.misclassifications = []
        self.converged_epoch = None
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.misclassifications = []
        
        for epoch in range(self.max_epochs):
            misclassified = 0
            for i in range(n_samples):
                linear_output = np.dot(X[i], self.weights) + self.bias
                if y[i] * linear_output <= 0:
                    self.weights += self.learning_rate * y[i] * X[i]
                    self.bias += self.learning_rate * y[i]
                    misclassified += 1
            self.misclassifications.append(misclassified)
            if misclassified == 0:
                self.converged_epoch = epoch + 1
                break
        
        if self.converged_epoch is None:
            self.converged_epoch = self.max_epochs
            
    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return np.where(linear_output >= 0, 1, -1)

In [ ]:
def plot_decision_boundary(X_train, y_train, X_test, y_test, model, dataset_name):
    plt.figure(figsize=(10, 8))
    x_min = min(X_train[:, 0].min(), X_test[:, 0].min()) - 1
    x_max = max(X_train[:, 0].max(), X_test[:, 0].max()) + 1
    y_min = min(X_train[:, 1].min(), X_test[:, 1].min()) - 1
    y_max = max(X_train[:, 1].max(), X_test[:, 1].max()) + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    plt.scatter(X_train[y_train==-1][:,0], X_train[y_train==-1][:,1],
                color='blue', marker='o', label='Train Class -1', alpha=0.6)
    plt.scatter(X_train[y_train==1][:,0], X_train[y_train==1][:,1],
                color='red', marker='o', label='Train Class 1', alpha=0.6)
    plt.scatter(X_test[y_test==-1][:,0], X_test[y_test==-1][:,1],
                color='blue', marker='x', label='Test Class -1', s=80)
    plt.scatter(X_test[y_test==1][:,0], X_test[y_test==1][:,1],
                color='red', marker='x', label='Test Class 1', s=80)
    plt.title(f'Decision Boundary for Dataset {dataset_name}')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.grid(True)
    plt.savefig(f'q3_decision_boundary_{dataset_name}.png')
    plt.show()

def plot_misclassifications(model, dataset_name):
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(model.misclassifications) + 1), model.misclassifications, marker='o')
    plt.title(f'Misclassified Samples per Epoch - Dataset {dataset_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Number of Misclassifications')
    plt.grid(True)
    plt.savefig(f'q3_misclassifications_{dataset_name}.png')
    plt.show()

In [ ]:
def process_dataset(X, y, dataset_name):
    print(f"\n--- Processing Dataset {dataset_name} ---")
    n_samples = len(X)
    train_size = int(0.7 * n_samples)
    X_train, y_train = X[:train_size], y[:train_size]
    X_test, y_test = X[train_size:], y[train_size:]
    
    model = Perceptron(learning_rate=0.01, max_epochs=300)
    model.fit(X_train, y_train)
    
    print(f"Convergence Epoch: {model.converged_epoch}")
    y_test_pred = model.predict(X_test)
    test_accuracy = np.mean(y_test_pred == y_test)
    print(f"Test Accuracy: {test_accuracy:.4f}")
    
    plot_misclassifications(model, dataset_name)
    plot_decision_boundary(X_train, y_train, X_test, y_test, model, dataset_name)

In [ ]:
# --- Q3 Main Execution ---

# Dataset A: well-separated (cov = I)
mean1 = [-3, -3]
cov1  = np.eye(2)
mean2 = [3, 3]
cov2  = np.eye(2)
X_A, y_A = generate_dataset(mean1, cov1, mean2, cov2, 200)
process_dataset(X_A, y_A, 'A')

# Dataset B: overlapping (cov = 3*I)
cov1_B = 3 * np.eye(2)
cov2_B = 3 * np.eye(2)
X_B, y_B = generate_dataset(mean1, cov1_B, mean2, cov2_B, 200)
process_dataset(X_B, y_B, 'B')